# Loss sensitivities over the globe: `unseen` and `base`

Where in the processor's latent space and in the model input do the globe-wide losses draw
their information from? This notebook compares checkpoints on the same date, for two scalars:

| scalar | loss |
|---|---|
| `unseen` | `UnseenObservationMaskedMSE` over the globe |
| `base` | `ObservationMaskedMSE` over the globe |

The single-cell scalar, with its radius of influence, is in `sensitivity_cell.ipynb`.

The gradients are computed by `sensitivity.py` on a GPU, in `nese-anemoi` (`sbatch submit_sensitivity.sh`).
The globe-wide scalars depend only on the date, so they are written to
`sensitivity/global/<t0>/<label>.zarr`. This notebook reads one date, set by `DATE` (or
`SENSITIVITY_DATE`), and runs in `nese`. Shared loading and drawing is in `sensitivity_plots.py`.

The gradients:
- **latent** is `|dL/dz|`, the norm over channels at each hidden (hpx6) node, where `z` is the processor input. With `latent_skip` the decoder sees `processor(z) + z`, so this is `J_procᵀ g + g`.
- **decoder** is `|g|`, the gradient at the decoder's hidden-node input: the skip path alone.
- **processor** is `|dL/dz − g|`, what reached `z` through the processor alone.
- **input** is `|dL/dx|`, summed over the variables of each kind (all sla missions, ...). `x` is the normalised, imputed input at each input hour.
- **encoder input** is the same at the encoder's input: each field's FieldEncoder latent (8 channels, at every cell, observed or not) and the passthrough inputs.

Maps are `log10` of the sensitivity over its maximum, clipped at `LOG_FLOOR`, so the checkpoints are compared by shape, not magnitude.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from sensitivity_plots import NOISE_FLOOR
from sensitivity_plots import draw
from sensitivity_plots import draw_seen
from sensitivity_plots import draw_solid
from sensitivity_plots import load
from sensitivity_plots import map_axes
from sensitivity_plots import normalised_log
from sensitivity_plots import plot_shares

# t0 of the run of sensitivity.py to read; submit_sensitivity_notebook.sh sets it per date
DATE = os.environ.get("SENSITIVITY_DATE", "2020-09-13T12")
CHECKPOINTS = ["v2p2", "v0", "control"]
SCALARS = ["unseen", "base"]

In [ ]:
sens = load(f"global/{DATE}", CHECKPOINTS)
print(f"t0 {sens.attrs['t0']} -> target {sens.attrs['target_time']}")
print(f"precision {sens.attrs.get('precision')}")
sens[["loss", "loss_scale"]].to_dataframe()

## Sensitivity share by input type

Left sums every variable of a kind. Middle divides by the kind's number of input variables (22
per mission kind, 9 static, 3 time-varying forcing). Count columns enter only through the observed
mask (`counts > 0`), so their gradient is zero. Right is taken at the encoder's input instead:
each field's FieldEncoder latent (8 channels) and the passthrough inputs (1 channel per
variable), divided by channel count. All gradients are with respect to normalised inputs, so the
normalisations also enter the comparison.

In [ ]:
for scalar in SCALARS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 3.5), constrained_layout=True)
    plot_shares(axes, sens.sel(scalar=scalar))
    fig.suptitle(scalar)

## Processor input

Total, skip path and processor path, each relative to the total's maximum, so the share of each
path is visible.

In [ ]:
paths = {"latent_sensitivity": "total", "decoder_sensitivity": "decoder (skip)", "processor_sensitivity": "processor"}

for scalar in SCALARS:
    sensitivity = sens.sel(scalar=scalar)
    reference = sensitivity["latent_sensitivity"].max("hidden")
    fig, axes = map_axes(len(CHECKPOINTS), len(paths))
    for row, label in enumerate(CHECKPOINTS):
        for column, (name, title) in enumerate(paths.items()):
            values = np.log10(sensitivity[name] / reference).clip(min=-4).sel(checkpoint=label)
            image = draw(axes[row, column], values)
            axes[row, column].set_title(f"{scalar}, {label}: {title}")
    fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|grad| / max total)")

## Model input by input hour

Summed over all variables. Grey dots are cells where sla was observed in that input bin.

In [ ]:
for scalar in SCALARS:
    total = normalised_log(sens["input_sensitivity"].sel(scalar=scalar).sum("kind"), ["cell", "input_hour"])
    fig, axes = map_axes(len(CHECKPOINTS), len(sens["input_hour"]))
    for row, label in enumerate(CHECKPOINTS):
        for column, hour in enumerate(sens["input_hour"].values):
            image = draw(axes[row, column], total.sel(checkpoint=label, input_hour=hour))
            draw_seen(axes[row, column], sens["sla_seen"].sel(input_hour=hour))
            axes[row, column].set_title(f"{scalar}, {label}: input hour {hour}")
    fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|dL/dx| / max)")

## Model input by kind, last input hour

Count kinds are left out: their gradient is zero by construction.

In [ ]:
kinds = ["sla", "ugxt", "vgxt", "swh", "time_forcing", "static"]

for scalar in SCALARS:
    last = sens["input_sensitivity"].sel(scalar=scalar, input_hour=sens["input_hour"].max(), kind=kinds)
    last = normalised_log(last, "cell")
    fig, axes = map_axes(len(CHECKPOINTS), len(kinds))
    for row, label in enumerate(CHECKPOINTS):
        for column, kind in enumerate(kinds):
            image = draw(axes[row, column], last.sel(checkpoint=label, kind=kind))
            axes[row, column].set_title(f"{scalar}, {label}: {kind}")
    fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|dL/dx| / max)")

## Where the fine tuning moved the sensitivity

The ratio of the shape-normalised input sensitivity, each later run over v2p2, summed over
kinds and input hours. Above 0 means a larger share of the sensitivity sits there after fine
tuning. Cells below `NOISE_FLOOR` of the maximum in either run, including exact zeros, are
mid grey, since their ratio could be numerical.

In [ ]:
shape = sens["input_total"] / sens["input_total"].sum("cell")
reference, others = CHECKPOINTS[0], CHECKPOINTS[1:]
ratio = np.log10(shape.sel(checkpoint=others) / shape.sel(checkpoint=reference))
amplitude = sens["input_total"] / sens["input_total"].max("cell")
noisy = (amplitude.sel(checkpoint=others) < NOISE_FLOOR) | (amplitude.sel(checkpoint=reference) < NOISE_FLOOR)

fig, axes = map_axes(len(others), len(SCALARS))
for row, label in enumerate(others):
    for column, scalar in enumerate(SCALARS):
        values, noise = ratio.sel(checkpoint=label, scalar=scalar), noisy.sel(checkpoint=label, scalar=scalar)
        image = draw(axes[row, column], values.where(~noise, drop=True), cmap="RdBu_r", vmin=-1, vmax=1)
        draw_solid(axes[row, column], noise.where(noise, drop=True), color="0.55")
        axes[row, column].set_title(f"{scalar}: log10({label} / {reference})")
fig.colorbar(image, ax=axes, shrink=0.6)